# 02 — Data Quality, Validation & Cleaning

Demonstrates the requirement to validate and clean institutional data to a high level of accuracy and integrity.

In [1]:
from pathlib import Path
import pandas as pd

ROOT = Path("..")
students = pd.read_csv(ROOT / "data/raw/students.csv")
enrollment = pd.read_csv(ROOT / "data/raw/enrollments.csv")
financial = pd.read_csv(ROOT / "data/raw/financial_aid.csv")

quality = pd.DataFrame({
    "check": [
        "Duplicate student IDs",
        "Duplicate enrollment keys",
        "Missing gender",
        "Credits completed > attempted",
        "Missing aid amount"
    ],
    "issues_found": [
        students.duplicated("student_id").sum(),
        enrollment.duplicated(["student_id","term_id","course_id"]).sum(),
        students["gender"].isna().sum(),
        (pd.to_numeric(enrollment["credits_completed"], errors="coerce") >
         pd.to_numeric(enrollment["credits_attempted"], errors="coerce")).sum(),
        financial["aid_amount"].isna().sum()
    ]
})
quality

,check,issues_found
0,Duplicate student IDs,35
1,Duplicate enrollment keys,120
2,Missing gender,168
3,Credits completed > attempted,45
4,Missing aid amount,75


In [2]:
program_map = {
    "BUS ADMIN": "Business Administration",
    "Health Admin": "Healthcare Administration"
}
students["program"] = students["program"].replace(program_map)
students["gender"] = students["gender"].fillna("Not reported")
students = students.drop_duplicates("student_id")

enrollment["withdrawal_flag"] = (
    enrollment["withdrawal_flag"]
    .replace({"Y":1, "N":0, "1":1, "0":0})
    .astype(int)
)
enrollment = enrollment.drop_duplicates(["student_id","term_id","course_id"])
enrollment["credits_attempted"] = pd.to_numeric(enrollment["credits_attempted"], errors="coerce")
enrollment["credits_completed"] = pd.to_numeric(enrollment["credits_completed"], errors="coerce")
bad_credit = enrollment["credits_completed"] > enrollment["credits_attempted"]
enrollment.loc[bad_credit, "credits_completed"] = enrollment.loc[bad_credit, "credits_attempted"]

print("Remaining duplicate student IDs:", students.duplicated("student_id").sum())
print("Remaining bad credit rows:", (enrollment["credits_completed"] > enrollment["credits_attempted"]).sum())

Remaining duplicate student IDs: 0
Remaining bad credit rows: 0


/tmp/ipykernel_695/2056336391.py:11: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  .replace({"Y":1, "N":0, "1":1, "0":0})


Ambiguous exceptions should be routed to a review file rather than silently overwritten when the business meaning cannot be determined from source-system rules.